# Fine-Tuning DistilBERT for Token Classification (POS Tagging & Chunking)

In [ ]:
!pip install transformers datasets seqeval

In [ ]:

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report, f1_score


## Load Dataset (CoNLL-2003)

In [ ]:

dataset = load_dataset("conll2003")
dataset


In [ ]:

label_list = dataset["train"].features["ner_tags"].feature.names
label_list


## Tokenization & Label Alignment

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


In [ ]:

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## Model Setup

In [ ]:

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)


## Training

In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)


In [ ]:

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions)
    }


In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


In [ ]:

trainer.train()


## Evaluation

In [ ]:

trainer.evaluate()


## Inference

In [ ]:

sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True)

outputs = model(**inputs).logits
predictions = np.argmax(outputs.detach().numpy(), axis=2)

predicted_labels = [label_list[p] for p in predictions[0]]

list(zip(tokens, predicted_labels[:len(tokens)]))



## Comparison: POS Tagging vs Chunking

- POS Tagging: Assigns grammatical labels (NN, VB, etc.)
- Chunking: Groups words into phrases (NP, VP)

## Observations
- Label alignment is tricky due to subword tokenization
- Chunking is more complex than POS tagging
- Transformer models perform well with proper preprocessing
